In [ ]:
from pathlib import Path
import os
import sys
import subprocess

# Change PROJECT_DIR only if your repository is somewhere else.
CANDIDATE_FOLDERS = [
    Path.cwd(),
    Path('F:/isaaclab/prototype-thesis'),
    Path('/mnt/f/isaaclab/prototype-thesis'),
]
PROJECT_DIR = next((p.resolve() for p in CANDIDATE_FOLDERS
                    if (p / 'scripts/train_fusion.py').is_file() and (p / 'dms').is_dir()), None)
assert PROJECT_DIR is not None, (
    'Repository not found. Edit CANDIDATE_FOLDERS above to point to the folder containing scripts/ and dms/.')
os.chdir(PROJECT_DIR)

DATA_YAML = PROJECT_DIR / 'datasets/driver_state/data.yaml'
DATA_MANIFEST = PROJECT_DIR / 'datasets/driver_state/split_manifest.csv'
ORIGINAL_WEIGHTS = (PROJECT_DIR / 'runs/quick_driver_state_12000_e35'
                    / 'yolo11n/weights/best.pt')
FACE_MODEL = PROJECT_DIR / 'models/face_landmarker.task'

FUSION_TABLE = PROJECT_DIR / 'datasets/fusion/esca_phase2_train_val.csv'
FUSION_MODEL = PROJECT_DIR / 'models/esca_fusion_phase2.json'
RUN_ROOT = PROJECT_DIR / 'results/esca_phase2_notebook'
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print('REPOSITORY:', PROJECT_DIR)
print('NOTEBOOK PYTHON:', sys.executable)
print('ENVIRONMENT:', 'WSL/Linux' if os.name != 'nt' else 'Windows')
print('YOLO checkpoint:', ORIGINAL_WEIGHTS)


REPOSITORY: F:\isaaclab\prototype-thesis
NOTEBOOK PYTHON: f:\isaaclab\env_isaaclab\Scripts\python.exe
ENVIRONMENT: Windows
YOLO checkpoint: F:\isaaclab\prototype-thesis\runs\quick_driver_state_12000_e35\yolo11n\weights\best.pt


## 1. Progress logging and Python environment

In [ ]:
import csv
import json
import hashlib
import shutil
import time
import random
import collections
from datetime import datetime

START = time.monotonic()

def log(message, level='INFO'):
    print(f'[{datetime.now():%H:%M:%S}] [+{time.monotonic()-START:,.0f}s] [{level}] {message}', flush=True)

def section(message):
    print('\n' + '=' * 76, flush=True)
    log(message, 'STEP')
    print('=' * 76, flush=True)

def run_verbose(*items, allow_failure=False):
    args = [str(x) for x in items]
    section('Running ' + ' '.join(args))
    started = time.monotonic()
    with subprocess.Popen(args, cwd=PROJECT_DIR, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                          errors='replace', bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        status = process.wait()
    log(f'Exit {status}; elapsed {(time.monotonic()-started)/60:.1f} minutes',
        'SUCCESS' if status == 0 else 'ERROR')
    if status and not allow_failure:
        raise RuntimeError(f'Command failed (exit code {status}). Read its output above.')
    return status

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def exists_or_explain(path, purpose):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Missing {purpose}: {path}. Check the paths in cell 1.')
    return path

section('Checking repository and Python')
run_verbose('git', 'rev-parse', '--short', 'HEAD', allow_failure=True)
run_verbose(sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt')

try:
    import cv2
    import yaml
    import numpy as np
    import torch
    import mediapipe
    from ultralytics import YOLO
except ImportError as exc:
    raise RuntimeError('Missing Python package. Set INSTALL_REQUIREMENTS=True in cell 1, rerun cells 1–2.') from exc
print('Python:', sys.version.split()[0])
print('OpenCV:', cv2.__version__, 'Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Repository requirements and core imports: OK')



[19:41:46] [+0s] [STEP] Checking repository and Python

[19:41:46] [+0s] [STEP] Running git rev-parse --short HEAD
467ec50
[19:41:46] [+0s] [SUCCESS] Exit 0; elapsed 0.0 minutes
Python: 3.12.13
OpenCV: 5.0.0 Torch: 2.11.0+cu130
CUDA available: True
Repository requirements and core imports: OK


## 2. Check existing models, datasets and class names

In [ ]:
section('Checking files from training.ipynb')
for path, purpose in [
    (ORIGINAL_WEIGHTS, 'the 4-class checkpoint trained in your old notebook'),
    (DATA_YAML, 'driver-state YOLO data.yaml'),
    (PROJECT_DIR / 'scripts/build_fusion_dataset.py', 'fusion data builder'),
    (PROJECT_DIR / 'scripts/train_fusion.py', 'fusion trainer'),
    (PROJECT_DIR / 'scripts/export_android_config.py', 'Android configuration exporter'),
]:
    exists_or_explain(path, purpose)
    log(f'FOUND: {path.relative_to(PROJECT_DIR)}')

if not FACE_MODEL.is_file():
    log('MediaPipe face_landmarker.task missing. Downloading repository models...', 'WARNING')
    run_verbose(sys.executable, 'scripts/download_models.py')
exists_or_explain(FACE_MODEL, 'MediaPipe face_landmarker.task')

EXPECTED_CLASSES = ['eye_closed', 'eye_open', 'phone', 'yawn']
old_model = YOLO(str(ORIGINAL_WEIGHTS))
actual_classes = ([old_model.names[i] for i in sorted(old_model.names)]
                  if isinstance(old_model.names, dict) else list(old_model.names))
print('YOLO classes:', actual_classes)
assert actual_classes == EXPECTED_CLASSES, (
    'Wrong model')
print('YOLO classes verified')

# Fix a stale Windows path only if the dataset's real image folders already exist here.
raw_yaml = yaml.safe_load(DATA_YAML.read_text(encoding='utf-8'))
root_field = Path(str(raw_yaml.get('path', '')))
if not root_field.is_absolute():
    root_field = DATA_YAML.parent / root_field if raw_yaml.get('path') else DATA_YAML.parent
real_root = (DATA_YAML.parent / root_field).resolve() if not root_field.is_absolute() else root_field
if not (real_root / 'images/train').is_dir():
    proposed = DATA_YAML.parent.resolve()
    if (proposed / 'images/train').is_dir():
        backup = DATA_YAML.with_suffix('.yaml.before_esca_phase2.bak')
        if not backup.exists():
            shutil.copy2(DATA_YAML, backup)
        raw_yaml['path'] = str(proposed)
        DATA_YAML.write_text(yaml.safe_dump(raw_yaml, sort_keys=False), encoding='utf-8')
        log(f'Corrected obsolete YAML dataset root. Backup: {backup}', 'WARNING')
    else:
        raise FileNotFoundError('Cannot find the dataset images. Restore datasets/driver_state/images/train first.')

print('DATASET YAML:', DATA_YAML)
print('DATASET SPLIT KEYS:', [key for key in ('train', 'val', 'test') if key in raw_yaml])



[19:42:22] [+36s] [STEP] Checking files from training.ipynb
[19:42:22] [+36s] [INFO] FOUND: runs\quick_driver_state_12000_e35\yolo11n\weights\best.pt
[19:42:22] [+36s] [INFO] FOUND: datasets\driver_state\data.yaml
[19:42:22] [+36s] [INFO] FOUND: scripts\build_fusion_dataset.py
[19:42:22] [+36s] [INFO] FOUND: scripts\train_fusion.py
[19:42:22] [+36s] [INFO] FOUND: scripts\export_android_config.py
YOLO classes: ['eye_closed', 'eye_open', 'phone', 'yawn']
YOLO classes verified. This notebook reuses your trained checkpoint unless RETRAIN_YOLO=True.
DATASET YAML: F:\isaaclab\prototype-thesis\datasets\driver_state\data.yaml
DATASET SPLIT KEYS: ['train', 'val', 'test']


## 3. Audit class coverage, source groups and labels

In [7]:
section('Counting images and bounding boxes without changing the dataset')
import importlib
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
from benchmark_yolo import load_split, label_path

split_counts = {}
examples_by_class = {}
for split in ('train', 'val', 'test'):
    images, names = load_split(str(DATA_YAML), split)
    ordered_names = [names[i] for i in sorted(names)]
    assert ordered_names == EXPECTED_CLASSES, f'{split}: wrong dataset class names {ordered_names}'
    boxes, image_counts = collections.Counter(), collections.Counter()
    sample = collections.defaultdict(list)
    malformed = []
    for img in images:
        label = label_path(img)
        if not label.exists():
            continue  # may be an intentionally empty/background image
        seen = set()
        for number, line in enumerate(label.read_text(encoding='utf-8').splitlines(), 1):
            if not line.strip():
                continue
            pieces = line.split()
            try:
                cls = int(pieces[0])
                coords = [float(v) for v in pieces[1:]]
                assert len(pieces) == 5 and cls in names
                assert all(0 <= v <= 1 for v in coords) and min(coords[2:]) > 0
            except (ValueError, AssertionError, IndexError):
                malformed.append(f'{label.name}:{number}  {line[:80]}')
                continue
            role = names[cls]
            boxes[role] += 1
            seen.add(role)
        for role in seen:
            image_counts[role] += 1
            if len(sample[role]) < 300:
                sample[role].append(str(img))
    split_counts[split] = {
        'images': len(images),
        'boxes': {role: boxes[role] for role in EXPECTED_CLASSES},
        'images_with_role': {role: image_counts[role] for role in EXPECTED_CLASSES},
        'malformed_count': len(malformed),
        'malformed_examples': malformed[:15],
    }
    examples_by_class[split] = sample
    print(f'{split.upper():5} images={len(images):6}  boxes={split_counts[split]["boxes"]}')
    if malformed:
        log(f'{split}: {len(malformed)} malformed bounding-box lines. Review before training.', 'WARNING')
    for role in EXPECTED_CLASSES:
        if boxes[role] == 0:
            log(f'{split}: NO {role} labels - class cannot be evaluated properly in this split.', 'WARNING')

known_groups = collections.defaultdict(set)
unknown_group_rows = 0
if DATA_MANIFEST.is_file():
    with DATA_MANIFEST.open(encoding='utf-8-sig', newline='') as stream:
        for row in csv.DictReader(stream):
            group = str(row.get('group') or '').strip()
            split = str(row.get('split') or '').lower().strip()
            split = {'valid':'val', 'validation':'val'}.get(split, split)
            if group and split in {'train', 'val', 'test'}:
                known_groups[group].add(split)
            else:
                unknown_group_rows += 1
    crossing = {group: sorted(splits) for group, splits in known_groups.items() if len(splits) > 1}
    print('Group overlap across splits:', crossing if crossing else 'None in known groups')
    print('Manifest rows with unknown/missing group:', unknown_group_rows)
    if crossing:
        log('GROUP LEAKAGE FOUND. Fix independently verified driver/person splits before final scores.', 'WARNING')
else:
    crossing = {}
    log('No split_manifest.csv. Cannot verify participant/source leakage.', 'WARNING')

DATA_AUDIT = RUN_ROOT / 'class_coverage_and_groups.json'
DATA_AUDIT.write_text(json.dumps({'source': str(DATA_YAML), 'splits': split_counts,
                                  'cross_split_groups': crossing, 'unknown_group_rows': unknown_group_rows},
                                 indent=2), encoding='utf-8')
print('Saved:', DATA_AUDIT)



[19:43:32] [+107s] [STEP] Counting images and bounding boxes without changing the dataset
TRAIN images= 29639  boxes={'eye_closed': 15387, 'eye_open': 33565, 'phone': 7104, 'yawn': 2518}
VAL   images=  7728  boxes={'eye_closed': 6304, 'eye_open': 6388, 'phone': 0, 'yawn': 1036}
[19:43:37] [+112s] [WARNING] val: NO phone labels - class cannot be evaluated properly in this split.
TEST  images=  6541  boxes={'eye_closed': 3057, 'eye_open': 9222, 'phone': 1575, 'yawn': 86}
Group overlap across splits: None in known groups
Manifest rows with unknown/missing group: 0
Saved: F:\isaaclab\prototype-thesis\results\esca_phase2_notebook\class_coverage_and_groups.json


In [8]:
section('Optional full dataset audit')

args = [sys.executable, 'scripts/audit_dataset.py', '--data', DATA_YAML.parent,
        '--out', RUN_ROOT / 'deep_audit']
if DATA_MANIFEST.is_file():
    args += ['--groups', DATA_MANIFEST]
code = run_verbose(*args, allow_failure=True)
print('Full audit report:', RUN_ROOT / 'deep_audit/audit.md')
if code:
    log('Full audit returned an issue status. Read audit.md before claiming leakage-free data.', 'WARNING')



[19:44:13] [+147s] [STEP] Optional full dataset audit

[19:44:13] [+147s] [STEP] Running f:\isaaclab\env_isaaclab\Scripts\python.exe scripts/audit_dataset.py --data F:\isaaclab\prototype-thesis\datasets\driver_state --out F:\isaaclab\prototype-thesis\results\esca_phase2_notebook\deep_audit --groups F:\isaaclab\prototype-thesis\datasets\driver_state\split_manifest.csv
  train: 0/29639
  train: 2000/29639
  train: 4000/29639
  train: 6000/29639
  train: 8000/29639
  train: 10000/29639
  train: 12000/29639
  train: 14000/29639
  train: 16000/29639
  train: 18000/29639
  train: 20000/29639
  train: 22000/29639
  train: 24000/29639
  train: 26000/29639
  train: 28000/29639
  val: 0/7728
  val: 2000/7728
  val: 4000/7728
  val: 6000/7728
  test: 0/6541
  test: 2000/6541
  test: 4000/6541
  test: 6000/6541
# Dataset audit: F:\isaaclab\prototype-thesis\datasets\driver_state

## Problems

- **strong class imbalance in train (largest/smallest class = 13x)**

## Splits

| Split | Images | Backgr

## 4. Train fusion model

In [13]:
section('Build fusion features from TRAIN and VALIDATION only')

REBUILD_FUSION_FEATURES = True

if REBUILD_FUSION_FEATURES or not FUSION_TABLE.is_file():
    cmd = [sys.executable, 'scripts/build_fusion_dataset.py',
            '--data', DATA_YAML, '--out', FUSION_TABLE,
            '--splits', 'train', 'val', 'test']
    if DATA_MANIFEST.is_file():
        cmd += ['--groups', DATA_MANIFEST]
    run_verbose(*cmd)
else:
    log(f'Reusing fusion table: {FUSION_TABLE}')
exists_or_explain(FUSION_TABLE, 'fusion feature CSV')
with FUSION_TABLE.open(newline='', encoding='utf-8') as stream:
    reader = csv.DictReader(stream)
    targets = collections.defaultdict(lambda: collections.Counter())
    for row in reader:
        split = row['split']
        assert split in ('train', 'val', 'test'), (
            f'Unexpected fusion dataset split: {split}')
        for head in ('eye_closed', 'yawn'):
            if row.get(head) in ('0', '1'):
                targets[(split, head)][row[head]] += 1
for split in ('train','val','test'):
    for head in ('eye_closed','yawn'):
        print(f'{split}/{head}: negative={targets[(split,head)]["0"]}, positive={targets[(split,head)]["1"]}')
for head in ('eye_closed','yawn'):
    assert targets[('train',head)]['0'] >= 20 and targets[('train',head)]['1'] >= 20, (
        f'Not enough TRAIN examples for {head}. Inspect data labels and fusion extraction summary.')
    assert sum(targets[('val',head)].values()) > 0, f'No validation examples for {head}.'


[20:28:51] [+2,826s] [STEP] Build fusion features from TRAIN and VALIDATION only

[20:28:51] [+2,826s] [STEP] Running f:\isaaclab\env_isaaclab\Scripts\python.exe scripts/build_fusion_dataset.py --data F:\isaaclab\prototype-thesis\datasets\driver_state\data.yaml --out F:\isaaclab\prototype-thesis\datasets\fusion\esca_phase2_train_val.csv --splits train val test --groups F:\isaaclab\prototype-thesis\datasets\driver_state\split_manifest.csv
W0000 00:00:1794832132.880369    7108 face_landmarker_graph.cc:181] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1794832132.885752   30296 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794832132.896603   30296 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
  train

In [14]:
section('Fit fusion model, verify both output heads, and inspect report')

run_verbose(sys.executable, 'scripts/train_fusion.py',
            '--table', FUSION_TABLE, '--out', FUSION_MODEL,
            '--hidden', '16', '--seed', '0')
model_data = json.loads(exists_or_explain(FUSION_MODEL, 'trained fusion model').read_text(encoding='utf-8'))
heads = set(model_data.get('heads', {}))
assert {'eye_closed', 'yawn'} <= heads, (
    f'Fusion training is incomplete. Needed both eye_closed and yawn; got {sorted(heads)}.')
fusion_report = FUSION_MODEL.with_name(FUSION_MODEL.stem + '_report.md')
print('Fusion heads:', sorted(heads))
print('Fusion model:', FUSION_MODEL)
if fusion_report.is_file():
    print(fusion_report.read_text(encoding='utf-8')[:7500])


[20:40:45] [+3,539s] [STEP] Fit fusion model, verify both output heads, and inspect report

[20:40:45] [+3,539s] [STEP] Running f:\isaaclab\env_isaaclab\Scripts\python.exe scripts/train_fusion.py --table F:\isaaclab\prototype-thesis\datasets\fusion\esca_phase2_train_val.csv --out F:\isaaclab\prototype-thesis\models\esca_fusion_phase2.json --hidden 16 --seed 0
# Fusion model report

Table: `F:\isaaclab\prototype-thesis\datasets\fusion\esca_phase2_train_val.csv` - trained 2026-11-16T20:40:46


| Sign | Model kept | Test pictures (no/yes) | Rules F1 | Fusion F1 | Rules + fusion F1 |
|---|---|---|---|---|---|
| Eyes closed | mlp | [4620, 1591] | 1.000 | 0.998 | 0.998 |
| Yawning | mlp | [6264, 85] | 1.000 | 0.994 | 0.994 |

Rules = EAR / MAR and blendshape thresholds with population defaults (no calibration on single pictures).
Rules + fusion = the live program with the fusion model loaded (equal weights).

Model: F:\isaaclab\prototype-thesis\models\esca_fusion_phase2.json
Report: F:\isaa

## 5. Export Android candidate as LiteRT FP32, labels, config and SHA256 manifest
*put litert_export.py in the same folder as this .ipynb then run it in wsl*